<a href="https://colab.research.google.com/github/IgorForceBR0202/linguagens_programacao/blob/main/Atividade_Pr%C3%A1tica_Visualiza%C3%A7%C3%A3o_Geoespacial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Contexto do Problema**

Você está desenvolvendo um módulo de inteligência geográfica para mapear oportunidades imobiliárias e analisar a distribuição de propriedades nos municípios de Nova Iguaçu e Queimados. Utilizando uma base de dados de captação de imóveis, seu objetivo é criar um mapa interativo que permita à equipe de vendas visualizar a localização exata das propriedades, identificar concentrações de ofertas através de clusters e analisar o valor de mercado de forma espacial


In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}



In [ ]:
df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)


## **Parte 1: Inicialização e Marcadores Básicos**

Crie um mapa base (folium.Map) centralizado na coordenada média de todos os imóveis do DataFrame. Ajuste o nível de zoom inicial (zoom_start) para 12 e utilize o estilo de mapa padrão (OpenStreetMap).
Itere sobre as 5 primeiras linhas do DataFrame e adicione um marcador simples (folium.Marker) para cada imóvel. Configure o popup para exibir o tipo do imóvel e o valor de venda


In [ ]:
import folium

centro_lat = df_mapa['latitude'].mean()
centro_lon = df_mapa['longitude'].mean()

mapa = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=12
)

for i in range(5):
    imovel = df_mapa.iloc[i]

    folium.Marker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        popup=(
            f"Tipo: {imovel['tipo']}<br>"
            f"Valor de venda: R$ {imovel['valor_venda']:.2f}"
        )
    ).add_to(mapa)

mapa

## **Parte 2: Customização Visual com Marcadores Circulares**

Crie um novo mapa base. Desta vez, utilize folium.CircleMarker para representar todos os imóveis da base. 4. Configure os marcadores circulares com as seguintes regras de negócio:
Raio: Fixo em 8 pixels.
Cor de preenchimento: Azul para 'Nova Iguaçu' e Laranja para 'Queimados'.
Tooltip: Exiba o texto "Clique para detalhes" ao passar o mouse.


In [ ]:

mapa_circular = folium.Map(
    location=[
        df_mapa['latitude'].mean(),
        df_mapa['longitude'].mean()
    ],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, imovel in df_mapa.iterrows():


    if imovel['cidade'] == 'Nova Iguaçu':
        cor = 'blue'
    else:
        cor = 'orange'

    folium.CircleMarker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        tooltip='Clique para detalhes'
    ).add_to(mapa_circular)

mapa_circular